In [1]:
import os
import sys
import matplotlib
matplotlib.use("Qt5Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from shapely.ops import unary_union
from concurrent.futures import ProcessPoolExecutor

import pypsa

import random
import cartopy.io.shapereader as shpreader
from shapely.geometry import Point
from shapely.prepared import prep

from tqdm import tqdm
from functools import partialmethod
tqdm.__init__ = partialmethod(tqdm.__init__, disable=True)

import logging
logging.getLogger("pypsa").setLevel(logging.CRITICAL)
logging.getLogger("linopy").setLevel(logging.CRITICAL)

land_shp = shpreader.natural_earth(resolution="10m", category="physical", name="land")
land_geoms = list(shpreader.Reader(land_shp).geometries())
_land = prep(unary_union(land_geoms)) 

Below are configurations and a set of functions the algorithm will use.

In [2]:
new_bus = "New Battery Node" # Name of the new bus
# Scenario paths for different network configurations
scenario_paths = ("networks/SV2024_north-west.nc", "networks/WP2024_north-west.nc", "networks/SV2033_north-west.nc", "networks/WP2033_north-west.nc")
chosen = 2 # Index of the scenario to run
x, y = -8.5, 54.5 # Coordinates of the new bus
new_bus_v_nom = 110 # Voltage of the new bus

k_list = [1, 2, 3, 4, 5] # Connection counts to test
results = [] # List to store results for each connection count

POP_SIZE = 10       # Genetic algorithm population size
GENERATIONS = 5      # Iterations for the genetic algorithm
ELITE_SIZE = 3        # Top performers carried over unchanged each generation
MUTATION_STD = 0.15   # Degrees — roughly ~15km, tune relative to your bounding box size

#CPU_CORES_TO_USE = 4 # Number of CPU cores to use for parallel processing

ENFORCE_MAX_CONNECTION_DISTANCE = True  # Toggle: True = bounded version, False = original unbounded version
MAX_CONNECTION_KM = 100  # Only used when the toggle above is True


def get_candidate_buses(network, cand_x, cand_y):
    eligible = [
        bus for bus in network.buses.index
        if bus != new_bus and network.buses.at[bus, "v_nom"] == new_bus_v_nom
    ]
    if ENFORCE_MAX_CONNECTION_DISTANCE:
        eligible = [
            bus for bus in eligible
            if haversine_km(cand_x, cand_y,
                             network.buses.at[bus, "x"], network.buses.at[bus, "y"]) <= MAX_CONNECTION_KM
        ]
    return eligible

# Uses the percentage active to calculate the total curtailment in MWh for specified renewable carriers in the network.
def total_curtailment_mwh(network, renewable_carriers=("wind",)):
    gens = network.generators
    curtailable = gens[gens.carrier.isin(renewable_carriers)].index
    available = (
        network.generators_t.p_max_pu[curtailable]
        * gens.loc[curtailable, "p_nom_opt"]
    )
    dispatched = network.generators_t.p[curtailable]
    curtailed_mw = (available - dispatched).clip(lower=0)
    weighted = curtailed_mw.mul(network.snapshot_weightings.generators, axis=0)
    return weighted.sum().sum()

# Calculates the great-circle distance (in kms) between two points on the Earth specified by their longitude and latitude in decimal degrees.
def haversine_km(lon0, lat0, lon1, lat1):
    R = 6371.0
    p0, p1 = np.radians(lat0), np.radians(lat1)
    dphi = np.radians(lat1 - lat0)
    dlambda = np.radians(lon1 - lon0)
    a = np.sin(dphi / 2)**2 + np.cos(p0) * np.cos(p1) * np.sin(dlambda / 2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

# Calculates the percentage of renewable energy curtailment avoided by adding a new bus to the network, based on the total available renewable energy and the curtailment before and after the addition of the bus.
def surplus_waste_reduction_pct(network, curtailment_before, curtailment_after,
                                  renewable_carriers=("wind",)):
    gens = network.generators
    curtailable = gens[gens.carrier.isin(renewable_carriers)].index
    available = (
        network.generators_t.p_max_pu[curtailable]
        * gens.loc[curtailable, "p_nom_opt"]
    )
    total_available_mwh = available.mul(
        network.snapshot_weightings.generators, axis=0
    ).sum().sum()
    avoided_mwh = curtailment_before - curtailment_after
    return avoided_mwh / total_available_mwh * 100

# Checks if the given longitude and latitude coordinates are located on land using a preprocessed land geometry.
def is_on_land(lon, lat):
    return _land.contains(Point(lon, lat))

Setup some constants and preprocessed data.

In [3]:
# The network to compare against
n_baseline = pypsa.Network(scenario_paths[chosen])
n_baseline.optimize(solver_options={"log_to_console": False, "include_objective_constant": True})
curtailment_before = total_curtailment_mwh(n_baseline)

# Chosen scenario network, objective weighting adjusted to represent a full year (8760 hours) 
# given the representative period of a week (168 hours).
n_base = pypsa.Network(scenario_paths[chosen])
n_base.snapshot_weightings["objective"] *= 8760 / 168

# 100 €/MWh penalty for curtailment to encourage the optimizer to reduce curtailment for wind generators.
curtailable = n_base.generators.index[n_base.generators.carrier == "wind"]
curtailment_penalty = 100
n_base.generators.loc[curtailable, "marginal_cost"] -= curtailment_penalty

# Define line cost and electrical parameters for the new connections (per MW per km, resistance and reactance per km).
line_cost_per_mw_km = 300
x_per_km = 0.35  # ohm/km
r_per_km = 0.12  # ohm/km

# Search bounding box, derived from existing buses at the target voltage 
# (connecting with different voltage would require a transformer, which is not considered here). 
# A margin is added to the bounding box to allow for some flexibility in the search area. 
# Margin is arbitrarily set to 0.2 degrees
candidate_voltage_buses = n_base.buses[n_base.buses.v_nom == new_bus_v_nom]
margin = 0.2
lon_min, lon_max = candidate_voltage_buses.x.min() - margin, candidate_voltage_buses.x.max() + margin
lat_min, lat_max = candidate_voltage_buses.y.min() - margin, candidate_voltage_buses.y.max() + margin

/home/icerydev/HackathonEnv/Code/Hackathons/TPSA_Hackathon/.venv/lib/python3.14/site-packages/pypsa/network/io.py:2082: FutureWarning: pandas infers the `str` dtype for string data since its version 3.0. PyPSA still converts it back to numpy object dtype on import, but will keep it from PyPSA 2.0 on. Set `pypsa.options.api.legacy_string_dtype` explicitly to suppress this warning.
  new_static = _coerce_string_dtypes(new_static)
/tmp/ipykernel_6890/383600167.py:3: FutureWarning: The default value of `include_objective_constant` will change from True to False in version 2.0. Set `include_objective_constant` explicitly to suppress this warning. Using False improves LP numerical conditioning by not including the objective constant as a variable.
  n_baseline.optimize(solver_options={"log_to_console": False, "include_objective_constant": True})


Here is the main algorithm that both uses the Genetic Algorithm and the candidate elimination for battery siting.
- Last run took 9m 3s, which is mainly due to the fact that we check each node for multiple amounts of connections.
- Time complexity $O(G⋅P⋅K⋅S)$

In [ ]:
# Solver option used for every optimization to suppress the solver's console log.
QUIET_SOLVER_OPTIONS = {"log_to_console": False, "include_objective_constant": True}

total_runs = POP_SIZE * GENERATIONS * len(k_list) * 2  # Total number of optimization runs for progress tracking
current_run = 0  # Counter for the current optimization run
# Sample a random (lon, lat) inside the bounding box that's on land.
def random_land_point():
    for _ in range(50):
        cand_x = random.uniform(lon_min, lon_max)
        cand_y = random.uniform(lat_min, lat_max)
        if is_on_land(cand_x, cand_y):
            return cand_x, cand_y
    return x, y


def evaluate_location(cand_x, cand_y):
    global current_run
    if not is_on_land(cand_x, cand_y):
        return None, -1e9  # Rejection if the candidate point is not on land

    scores = {}
    for k in k_list:
        # Start each k evaluation from the unmodified base network.
        n_cand = n_base.copy()
        n_cand.add("Bus", new_bus, x=cand_x, y=cand_y, v_nom=new_bus_v_nom)
        n_cand.add("StorageUnit",
                   f"Battery at {new_bus}",
                   bus=new_bus,
                   p_nom_extendable=True,
                   p_nom_min=0,
                   p_nom_max=2_000, # Battery maximum power capacity in MW
                   capital_cost=75_000, # Annual battery cost in €/MW
                   marginal_cost=0.1,
                   efficiency_store=0.95,
                   efficiency_dispatch=0.95,
                   max_hours=4,
                   cyclic_state_of_charge=True)

        # candidate_buses = [
        #     bus for bus in n_cand.buses.index
        #     if bus != new_bus and n_cand.buses.at[bus, "v_nom"] == new_bus_v_nom
        # ]
        candidate_buses = get_candidate_buses(n_cand, cand_x, cand_y)
        for bus in candidate_buses:
            x0, y0 = n_cand.buses.at[bus, "x"], n_cand.buses.at[bus, "y"]
            length = haversine_km(cand_x, cand_y, x0, y0)
            n_cand.add("Line",
                       f"{new_bus} - {bus}",
                       bus0=new_bus, bus1=bus,
                       x=x_per_km * length,
                       r=r_per_km * length,
                       s_nom_extendable=True,
                       s_nom_min=0,
                       s_nom_max=200,
                       capital_cost=length * line_cost_per_mw_km,
                       length=length)

        # First optimization ranks the candidate connections for this location.
        print(f"Running optimization {(current_run := current_run + 1)}/{total_runs} for location ({cand_x:.4f}, {cand_y:.4f}) with k={k}")
        n_cand.optimize(**QUIET_SOLVER_OPTIONS)

        built = {}
        for line_name in n_cand.lines.index:
            if line_name.startswith(new_bus):
                s_opt = n_cand.lines.at[line_name, "s_nom_opt"]
                if s_opt > 0.01:
                    built[line_name.split(" - ")[1]] = s_opt
        ranked_buses = sorted(built, key=built.get, reverse=True)

        # Remove all temporary first-pass lines before adding only this k's lines.
        first_pass_lines = n_cand.lines.index[n_cand.lines.index.str.startswith(new_bus)]
        n_cand.remove("Line", first_pass_lines)

        top_buses = ranked_buses[:k]
        for bus in top_buses:
            x0, y0 = n_cand.buses.at[bus, "x"], n_cand.buses.at[bus, "y"]
            length = haversine_km(cand_x, cand_y, x0, y0)
            n_cand.add("Line",
                       f"{new_bus} - {bus}",
                       bus0=new_bus, bus1=bus,
                       x=x_per_km * length,
                       r=r_per_km * length,
                       s_nom_extendable=True,
                       s_nom_min=0,
                       s_nom_max=200,
                       capital_cost=length * line_cost_per_mw_km,
                       length=length)

        print(f"Running optimization {(current_run := current_run + 1)}/{total_runs} for location ({cand_x:.4f}, {cand_y:.4f}) with k={k}")
        n_cand.optimize(**QUIET_SOLVER_OPTIONS)
        curtailment_after = total_curtailment_mwh(n_cand)
        scores[k] = surplus_waste_reduction_pct(
            n_cand, curtailment_before, curtailment_after
        )

    # Return the optimal k and its score for this candidate location.
    return max(scores.items(), key=lambda item: item[1])

def evaluate_location_wrapper(loc):
    return evaluate_location(*loc)

# Genetic Algorithm to find the best location for the new bus
population = [random_land_point() for _ in range(POP_SIZE)]
best_overall = (None, None, -1e9)  # (location, k, fitness)

for gen in range(GENERATIONS):
    scored = [
        (loc, *evaluate_location(*loc))
        for loc in population
    ]
    scored.sort(key=lambda item: item[2], reverse=True)

    if scored[0][2] > best_overall[2]:
        best_overall = scored[0]

    print(f"Generation {gen}: best fitness so far = {best_overall[2]:.3f}% "
          f"at {best_overall[0]} with k={best_overall[1]}")

    elites = [loc for loc, _, _ in scored[:ELITE_SIZE]]
    new_population = list(elites)  # elitism: carry the best forward unchanged

    while len(new_population) < POP_SIZE:
        parent_a, parent_b = random.sample(elites, 2)
        # crossover: blend the two parents' coordinates
        child_x = (parent_a[0] + parent_b[0]) / 2
        child_y = (parent_a[1] + parent_b[1]) / 2
        # mutation: small random nudge
        child_x += random.gauss(0, MUTATION_STD)
        child_y += random.gauss(0, MUTATION_STD)
        child_x = min(max(child_x, lon_min), lon_max)
        child_y = min(max(child_y, lat_min), lat_max)
        if not is_on_land(child_x, child_y):
            continue  # reject and retry rather than accepting a sea location
        new_population.append((child_x, child_y))

    population = new_population

best_x, best_y = best_overall[0]
best_k = best_overall[1]
print(f"\nGA result: best location = ({best_x:.4f}, {best_y:.4f}), "
      f"k = {best_k}, fitness = {best_overall[2]:.3f}%")

n = n_base.copy()
n.add("Bus", new_bus, x=best_x, y=best_y, v_nom=new_bus_v_nom)
n.add("StorageUnit",
      f"Battery at {new_bus}",
      bus=new_bus,
      p_nom_extendable=True,
      p_nom_min=0,
      p_nom_max=2_000,
      capital_cost=75_000,
      marginal_cost=0.1,
      efficiency_store=0.95,
      efficiency_dispatch=0.95,
      max_hours=4,
      cyclic_state_of_charge=True)

# candidate_buses = [
#     bus for bus in n.buses.index
#     if bus != new_bus and n.buses.at[bus, "v_nom"] == new_bus_v_nom
# ]
candidate_buses = get_candidate_buses(n, best_x, best_y)
for bus in candidate_buses:
    x0, y0 = n.buses.at[bus, "x"], n.buses.at[bus, "y"]
    length = haversine_km(best_x, best_y, x0, y0)
    n.add("Line",
          f"{new_bus} - {bus}",
          bus0=new_bus, bus1=bus,
          x=x_per_km * length,
          r=r_per_km * length,
          s_nom_extendable=True,
          s_nom_min=0,
          s_nom_max=200,
          capital_cost=length * line_cost_per_mw_km,
          length=length)

n.optimize()

built = {}
for line_name in n.lines.index:
    if line_name.startswith(new_bus):
        s_opt = n.lines.at[line_name, "s_nom_opt"]
        if s_opt > 0.01:
            built[line_name.split(" - ")[1]] = s_opt
ranked_buses = sorted(built, key=built.get, reverse=True)

n.model.solver_model = None
first_pass_lines = n.lines.index[n.lines.index.str.startswith(new_bus)]
n.remove("Line", first_pass_lines)
n_clean = n.copy()

# Rebuild and optimize only the GA-selected k network.
top_buses = ranked_buses[:best_k]
for bus in top_buses:
    x0, y0 = n_clean.buses.at[bus, "x"], n_clean.buses.at[bus, "y"]
    length = haversine_km(best_x, best_y, x0, y0)
    n_clean.add("Line",
                f"{new_bus} - {bus}",
                bus0=new_bus, bus1=bus,
                x=x_per_km * length,
                r=r_per_km * length,
                s_nom_extendable=True,
                s_nom_min=0,
                s_nom_max=200,
                capital_cost=length * line_cost_per_mw_km,
                length=length)

n_clean.optimize(**QUIET_SOLVER_OPTIONS)

battery_opt = n_clean.storage_units.at[f"Battery at {new_bus}", "p_nom_opt"]
curtailment_after = total_curtailment_mwh(n_clean)
pct_change = (curtailment_after - curtailment_before) / curtailment_before * 100
waste_reduction = surplus_waste_reduction_pct(
    n_clean, curtailment_before, curtailment_after
)
results = [{
    "K": best_k,
    "top_buses": top_buses,
    "battery_mw": battery_opt,
    "battery_mwh": battery_opt * 4,
    "curtailment_after_mwh": curtailment_after,
    "dispatch_down_pct_change": pct_change,
    "surplus_waste_reduction_pct": waste_reduction,
    "network": n_clean,
}]

# print(f"\n{'K':>3} | {'Battery (MW)':>13} | {'Battery (MWh)':>14} | {'Curtailment after (MWh)':>24} | {'Waste reduced %':>16}")
# print("-" * 90)
# for r in results:
#     print(f"{r['K']:>3} | {r['battery_mw']:>13.2f} | {r['battery_mwh']:>14.2f} | "
#           f"{r['curtailment_after_mwh']:>24.2f} | {r['surplus_waste_reduction_pct']:>16.2f}")

Running optimization 1/500 for location (-9.2966, 53.9895) with k=1
Running optimization 2/500 for location (-9.2966, 53.9895) with k=1
Running optimization 3/500 for location (-9.2966, 53.9895) with k=2
Running optimization 4/500 for location (-9.2966, 53.9895) with k=2
Running optimization 5/500 for location (-9.2966, 53.9895) with k=3
Running optimization 6/500 for location (-9.2966, 53.9895) with k=3
Running optimization 7/500 for location (-9.2966, 53.9895) with k=4
Running optimization 8/500 for location (-9.2966, 53.9895) with k=4
Running optimization 9/500 for location (-9.2966, 53.9895) with k=5
Running optimization 10/500 for location (-9.2966, 53.9895) with k=5
Running optimization 11/500 for location (-8.9300, 54.1980) with k=1
Running optimization 12/500 for location (-8.9300, 54.1980) with k=1
Running optimization 13/500 for location (-8.9300, 54.1980) with k=2
Running optimization 14/500 for location (-8.9300, 54.1980) with k=2
Running optimization 15/500 for location (-

/tmp/ipykernel_6890/891350043.py:177: FutureWarning: The default value of `include_objective_constant` will change from True to False in version 2.0. Set `include_objective_constant` explicitly to suppress this warning. Using False improves LP numerical conditioning by not including the objective constant as a variable.
  n.optimize()


Running HiGHS 1.15.1 (git hash: 04024d7): Copyright (c) 2026 under MIT licence terms
Includes third-party software components, see THIRD_PARTY_NOTICES.md for full details
LP linopy-problem-_hdgv6jv has 28418 rows; 11437 cols; 52274 nonzeros
Coefficient ranges:
  Matrix  [9e-01, 2e+02]
  Cost    [5e+00, 5e+05]
  Bound   [0e+00, 0e+00]
  RHS     [3e-02, 2e+03]
Presolving model
9285 rows, 8781 cols, 29588 nonzeros 0s
7724 rows, 7220 cols, 31195 nonzeros 0s
Dependent equations search running on 2677 equations with time limit of 1000.00s
Dependent equations search removed 0 rows and 0 nonzeros in 0.00s (limit = 1000.00s)
7213 rows, 6542 cols, 33278 nonzeros 0s
Presolve reductions: rows 7213(-21205); columns 6542(-4895); nonzeros 33278(-18996) 
Solving the presolved LP
Using dual simplex solver
  Iteration        Objective     Infeasibilities num(sum)
          0    -2.0281209216e-08 Ph1: 5107(4.98871e+06); Du: 0(2.4461e-10) 0.0s
       3805    -5.7675887024e+08 Pr: 0(0); Du: 0(7.17798e-10) 

/tmp/ipykernel_6890/891350043.py:208: FutureWarning: The default value of `include_objective_constant` will change from True to False in version 2.0. Set `include_objective_constant` explicitly to suppress this warning. Using False improves LP numerical conditioning by not including the objective constant as a variable.
  n_clean.optimize()


Running HiGHS 1.15.1 (git hash: 04024d7): Copyright (c) 2026 under MIT licence terms
Includes third-party software components, see THIRD_PARTY_NOTICES.md for full details
LP linopy-problem-nbut23_9 has 24370 rows; 10085 cols; 40666 nonzeros
Coefficient ranges:
  Matrix  [9e-01, 2e+02]
  Cost    [5e+00, 5e+05]
  Bound   [0e+00, 0e+00]
  RHS     [3e-02, 2e+03]
Presolving model
5083 rows, 7259 cols, 17656 nonzeros 0s
3873 rows, 6049 cols, 16589 nonzeros 0s
Dependent equations search running on 1965 equations with time limit of 1000.00s
Dependent equations search removed 0 rows and 0 nonzeros in 0.00s (limit = 1000.00s)
3813 rows, 5153 cols, 15911 nonzeros 0s
Presolve reductions: rows 3813(-20557); columns 5153(-4932); nonzeros 15911(-24755) 
Solving the presolved LP
Using dual simplex solver
  Iteration        Objective     Infeasibilities num(sum)
          0     4.8110607493e-09 Ph1: 672(758617); Du: 0(6.26525e-11) 0.0s
       2591    -5.7694221320e+08 Pr: 0(0); Du: 0(1.9604e-10) 0.1s



Print results.

In [15]:
# Summarize the single optimal solution selected by the genetic algorithm.
best_result = results[0]
best_K = best_result["K"]
n_best = best_result["network"]

label_width = 46
value_width = 14

def pline(label, value_str):
    print(f"{label:<{label_width}}{value_str:>{value_width}}")

print("Optimal battery placement solution")
print("=" * (label_width + value_width))
pline("Connection count (K):", f"{best_K}")
pline("Selected location (longitude):", f"{best_x:.4f}°")
pline("Selected location (latitude):", f"{best_y:.4f}°")
pline("Curtailment before installation:", f"{curtailment_before:,.2f} MWh")
pline("Curtailment after installation:", f"{best_result['curtailment_after_mwh']:,.2f} MWh")
pline("Dispatch down percentage change (Δ):",
     f"{((-1 + best_result['curtailment_after_mwh'] / curtailment_before) * 100):+.2f}%")
pline("Renewable waste reduction:", f"{best_result['surplus_waste_reduction_pct']:+.2f}%")
pline("Battery energy capacity:", f"{best_result['battery_mwh']:,.2f} MWh")

Optimal battery placement solution
Connection count (K):                                      4
Selected location (longitude):                      -7.7271°
Selected location (latitude):                       54.6547°
Curtailment before installation:               21,813.61 MWh
Curtailment after installation:                13,879.91 MWh
Dispatch down percentage change (Δ):                 -36.37%
Renewable waste reduction:                           +10.24%
Battery energy capacity:                        2,145.19 MWh


Draws map. 

---
Legend:
- Blue: Demand buses
- Orange: Newly placed battery
- Green: Wind power sources
- Red: Other power sources

In [ ]:
n_best.model.solver_model = None  # detach solved model before plotting/copying

bus_colours = pd.Series("lightgray", index=n_best.buses.index, dtype="object")

wind_buses = n_best.generators.loc[n_best.generators.carrier == "wind", "bus"]
other_generator_buses = n_best.generators.loc[n_best.generators.carrier != "wind", "bus"]
load_buses = n_best.loads["bus"]
battery_buses = n_best.storage_units["bus"]

wind_and_load_buses = set(wind_buses.unique()) & set(load_buses.unique())

bus_colours.loc[other_generator_buses.unique()] = "red"
bus_colours.loc[wind_buses.unique()] = "green"
bus_colours.loc[load_buses.unique()] = "blue"
bus_colours.loc[list(wind_and_load_buses)] = "purple"
bus_colours.loc[battery_buses.unique()] = "orange"

n_best.plot(bus_sizes=0.0025, margin=0.25, bus_colors=bus_colours)
plt.title(f"Optimal network topology — ({best_x:.3f}, {best_y:.3f}), K={best_K}")
plt.show()

qt.qpa.wayland: Wayland does not support QWindow::requestActivate()
